1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
 (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다

In [14]:
# from dotenv import load_dotenv
# load_dotenv()

#### STT (Speech To Text)

In [15]:
!pip install pyaudio

In [18]:
!pip install playsound==1.2.2

In [21]:
!pip install pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/10.6 MB 10.1 MB/s eta 0:00:01
   ------------------ --------------------- 5.0/10.6 MB 14.4 MB/s eta 0:00:01
   ------------------------------ --------- 8.1/10.6 MB 14.4 MB/s eta 0:00:01
   -------------------------------------- - 10.2/10.6 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 12.8 MB/s  0:00:00


- 사용할 함수들 모음

In [16]:


# import re


# from gtts import gTTS
# from sympy import re


# SYSTEM_PROMPT = """당신은 응급처치 전문 AI 도우미입니다.
# 사용자가 응급상황을 말하면 아래 형식으로 즉시 안내하세요.

# 상황 요약: (한 줄)
# 즉시 해야 할 일: (1~3가지)
# 단계별 응급처치: (구체적 수치 포함)
# 하지 말아야 할 것:
# 119 신고 시 전달할 내용:

# 규칙:
# - 반드시 한국어로 답변
# - 구체적인 수치(초, 회, cm) 포함
# - 쉬운 말 사용
# - 생명 위협 상황이면 119 신고를 가장 먼저 강조
# """

# def generate_response(user_input: str) -> str:
#     prompt = PromptTemplate.from_template(SYSTEM_PROMPT + "\n\n사용자 입력: {input}\n\n응답:")
#     llm = init_chat_model("openai:gpt-4.1-mini")
#     output_parser = StrOutputParser()

#     client = 'gpt-4.1-mini'

#     response = client.chat.completions.create(
#         model = "gpt-4.1-mini",
#         messages = [{
#             'role' : 'system',
#             'content' : SYSTEM_PROMPT
#         }],
#         temperature = 0.3,
#         max_tokens = 4096,
#         top_p = 1
#         )



# class InMemoryHistory(BaseChatMessageHistory, BaseModel):
#     '''사용자(Session)별 대화내용을 기록하는 클래스'''
#     messages: List[BaseMessage] = Field(default_factory=list)

#     def add_messages(self, messages: List[BaseMessage]) -> None:
#         self.messages.extend(messages)

#     def clear(self) -> None:
#         self.messages = []

# store = {}


# def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
#     if session_id not in store:
#         store[session_id] = InMemoryHistory()
#     return store[session_id]


# def text_to_speech(text: str):
#     clean = re.sub(r'[^\w\s.,!?가-힣ㄱ-ㅎㅏ-ㅣ]', ' ', text)
#     clean = re.sub(r'\s+', ' ', clean).strip()
#     tts_output = gTTS(text = clean, lang='ko', slow=False)





- 전체 코드

In [ ]:


import io
import re
from gtts import gTTS
import speech_recognition as sr
from IPython.display import display, Audio
from openai import OpenAI
from dotenv import load_dotenv
import time
from playsound import playsound
import pygame


load_dotenv()
recognizer = sr.Recognizer() # Recognizer 객체를 생성하여 음성 인식 기능을 사용할 준비를 함
api_messages = [] # 멀티턴 대화 기록을 위한 리스트
client = OpenAI()

SYSTEM_PROMPT = """당신은 응급처치 전문 AI 도우미입니다.

대화 방식:
1. 사용자가 응급상황을 말하면 핵심 응급처치 2~3가지만 간단히 알려주세요.
2. 그 다음 반드시 환자 상태를 확인하는 질문을 하나만 하세요.
3. 사용자가 답하면 그 답변에 맞게 추가 처치를 안내하고 또 다른 상태를 질문하세요.
4. 이런 식으로 대화를 이어가며 상황에 맞는 처치를 안내하세요.

예시:
사용자: 팔이 부러졌어요
AI: 팔을 움직이지 말고 부목으로 고정해주세요. 출혈이 있으면 깨끗한 천으로 눌러주세요.
    혹시 팔에 출혈이 있나요?
사용자: 네 피가 나요
AI: 깨끗한 천으로 출혈 부위를 강하게 눌러주세요. 5~10분간 유지하세요.
    출혈량이 많은가요, 적은가요?

규칙:
- 반드시 한국어로 답변
- 한 번에 질문은 하나만
- 짧고 명확하게
- 생명 위협 상황이면 119 신고를 가장 먼저 강조
"""

# LLM 응답 생성 함수
def generate_response(user_input: str) -> str:
    

    api_messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [{
            'role' : 'system',
            'content' : SYSTEM_PROMPT
        }] + api_messages[-6:],
        temperature = 0.3,
        max_tokens = 4096,
        top_p = 1
        )
    
    answer = response.choices[0].message.content
    api_messages.append({"role": "assistant", "content": answer})
    return answer






# TTS 함수
def text_to_speech(text: str):
    clean = re.sub(r'[^\w\s.,!?가-힣]', ' ', text)
    tts = gTTS(text=clean, lang='ko', slow=False)
    
    # 파일 저장 없이 메모리에서 바로 재생
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    buf.seek(0)
    
    pygame.mixer.init()
    pygame.mixer.music.load(buf)
    pygame.mixer.music.play()
    
    # 재생 끝날 때까지 대기
    while pygame.mixer.music.get_busy():
        pygame.time.Clock().tick(10)




'''----------------------------------------------------------------------------------'''

while True:
    with sr.Microphone() as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print("말씀하세요")

        try:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
            text = recognizer.recognize_google(audio, language='ko-KR')

            if text == '고마워':
                print("프로그램을 종료합니다.")
                break

            print(f"\n사용자: {text}")

            response = generate_response(text)
            print(f"AI: {response}\n")

            text_to_speech(response)

        except sr.WaitTimeoutError:
            print("시간 내에 말이 감지되지 않았습니다.")
        except sr.UnknownValueError:
            print("음성 인식에 실패했습니다.")
        except sr.RequestError as e:
            print(f"음성 인식 서비스 오류: {e}")

c:\Users\USER\anaconda3\envs\nlp_env\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
말씀하세요

사용자: 손가락이 잘렸어
AI: 우선 119에 바로 신고하세요.  
출혈이 심하면 깨끗한 천으로 강하게 눌러 지혈하고, 잘린 손가락은 깨끗한 거즈에 싸서 냉장 보관하세요.  
환자의 의식은 괜찮나요?

말씀하세요

사용자: 조금은 많이 아
AI: 의식이 있으니 다행입니다.  
출혈이 계속되면 압박 지혈을 유지하고, 손가락을 심장보다 높게 들어 올려주세요.  
호흡이나 맥박은 정상인가요?

말씀하세요

사용자: 맥박 정상 범위가 어느 정도
AI: 성인의 정상 맥박은 분당 60~100회입니다.  
맥박이 너무 빠르거나 느리면 즉시 119에 신고하세요.  
맥박은 정상인가요?

말씀하세요
음성 인식에 실패했습니다.
말씀하세요

사용자: 옹녀
AI: 맥박이 정상이라는 뜻으로 이해하겠습니다.  
출혈이 계속되면 압박 지혈을 유지하고, 환자가 안정되도록 눕히세요.  
환자가 어지럽거나 의식이 흐려지진 않나요?



KeyboardInterrupt: 

: 